# Модуль 2: Основы Pandas — Подготовка финансовых данных

Привет! Этот ноутбук посвящен знакомству с **Pandas** — главным инструментом AI-инженера для очистки и анализа таблиц.

Мы разберем базовые операции на примере грязных исторических котировок акций: научимся создавать таблицы, заполнять пропуски, фильтровать данные по условию и рассчитывать простейшие финансовые признаки.
Этот файл симулирует реальную задачу AI-инженера: обработку грязных финансовых данных для трейдинга. Внутри встроен мини-датасет, на котором показано, как загружать данные, искать пропуски, фильтровать аномалии и рассчитывать простейший торговый индикатор (скользящую среднюю).

In [ ]:
import pandas as pd
import numpy as np

### 1. Создание DataFrame
Обычно данные загружают из файлов с помощью `pd.read_csv()`, но сейчас мы создадим таблицу (DataFrame) вручную из словаря Python. Обратите внимание: в данных есть пропуски (`None`) и аномально высокая цена (выброс).

In [ ]:
# Сырые данные котировок
raw_data = {
    'Date': ['2026-09-14', '2026-09-15', '2026-09-16', '2026-09-17', '2026-09-18', '2026-09-19'],
    'Ticker': ['AAPL', 'AAPL', 'AAPL', 'AAPL', 'AAPL', 'AAPL'],
    'Close_Price': [175.5, None, 178.2, 999.0, 180.1, 182.4], # None — это пропуск, 999.0 — ошибка
    'Volume': [1000, 1200, 850, 1100, 950, 1300]
}

# Создаем DataFrame
df = pd.DataFrame(raw_data)

# Выводим первые строки таблицы
print("Исходная таблица:")
df

### 2. Очистка данных (Обработка пропусков)
Нейросети не умеют работать с пропусками (NaN). В Pandas найти их можно методом `.isna()`, а заполнить — методом `.fillna()`. Заполним пропущенную цену средним значением по столбцу.

In [ ]:
# Считаем среднюю цену (пропуски автоматически игнорируются)
mean_price = df['Close_Price'].mean()

# Заменяем пропуски средним значением прямо в исходной таблице
df['Close_Price'] = df['Close_Price'].fillna(mean_price)

print(f"Пропуски заполнены средним значением ({mean_price:.2f}):")
df

### 3. Фильтрация данных и удаление аномалий
Цена `999.0` — это явный технический сбой (выброс), который сломает обучение ИИ. Отфильтруем таблицу, оставив только реалистичные цены (например, меньше 500).

In [ ]:
# Оставляем в таблице только те строки, где цена меньше 500
df = df[df['Close_Price'] < 500]

# Сбрасываем индексы строк по порядку после удаления лишней строки
df = df.reset_index(drop=True)

print("Таблица после фильтрации аномалий:")
df

### 4. Создание новых признаков (Feature Engineering для трейдинга)
Для прогнозирования цен часто используют индикаторы. Создадим **SMA (Simple Moving Average)** — скользящую среднюю за 2 последних дня с помощью скользящего окна `.rolling()`.

In [ ]:
# Создаем новый столбец SMA_2 (среднее текущей и предыдущей строки)
df['SMA_2'] = df['Close_Price'].rolling(window=2).mean()

print("Итоговая таблица со сгенерированным индикатором SMA_2:")
df